# 深度学习课程设计报告

## 一、封面

- 课程名称：  深度学习课程设计
- 设计题目：  基于卷积神经网络的CIFAR-10图像分类
- 姓    名：  何曼希
- 学    号：  20234080122
- 班    级：  2023级
- 指导教师：  丁平尖
- 提交日期：  2026.6.18

## 二、摘要
本项目基于深度学习框架PyTorch，针对CIFAR-10数据集进行了图像分类任务的系统研究。CIFAR-10数据集包含10个类别的60000张32x32彩色图像，是一个广泛应用于计算机视觉领域的基准数据集。

本设计采用了两套模型架构：作为Baseline的SimpleCNN和作为最终模型的ResNet18。SimpleCNN由三层卷积层和两层全连接层构成，参数量较小，适合作为基准参考；ResNet18则通过残差连接解决了深层网络中的梯度消失问题，能够提取更丰富的特征表示。

实验过程中，对数据进行了随机裁剪、随机水平翻转等数据增强操作，并采用了Batch Normalization和Dropout技术来防止过拟合。通过20个epoch的训练，SimpleCNN在验证集上达到了78.36%的准确率，而ResNet18达到了90.56%的准确率，实现了12.20%的性能提升。项目还通过混淆矩阵、错误样本分析和特征图可视化等方法对模型行为进行了深入分析，验证了深度学习模型在图像分类任务中的有效性。

## 三、问题定义与需求分析

### 3.1 项目背景与意义

图像分类是计算机视觉领域的基础性任务，也是深度学习方法最早取得突破的方向之一。CIFAR-10数据集由Alex Krizhevsky等人于2009年发布，包含了飞机、汽车、鸟类、猫、鹿、狗、青蛙、马、船和卡车共10个类别的彩色图像。该数据集因其中等规模、类别丰富和图像分辨率适中的特点，成为评估计算机视觉算法性能的重要基准。

随着深度卷积神经网络（CNN）的发展，CIFAR-10的分类准确率已从传统方法的不足80%提升到95%以上。本研究旨在通过对比浅层CNN和深层残差网络在该任务上的性能表现，深入理解不同网络架构对图像分类任务的影响，为后续更复杂的视觉任务研究奠定基础

### 3.2 问题描述

输入定义：32×32像素的RGB彩色图像，像素值范围为0~255

输出定义：10个类别标签中的1个（plane, car, bird, cat, deer, dog, frog, horse, ship, truck）

任务类型：多分类任务（Multi-class Classification）

预期性能指标：

Top-1 准确率（Accuracy）：正确预测样本占总样本的比例

宏平均F1分数（Macro F1-Score）：各类别F1的算术平均

混淆矩阵（Confusion Matrix）：展示各类别的分类详细情况

## 四、数据集说明与预处理

### 4.1 数据来源与规模

数据来源：CIFAR-10公开数据集，由加拿大CIFAR机构发布，通过torchvision.datasets.CIFAR10加载

样本总量：60000张图像

训练集：50000张

测试集：10000张

类别分布：10个类别完全均衡，每个类别6000张图像

类别	样本数	类别	样本数
airplane	6000	frog	6000
automobile	6000	horse	6000
bird	6000	ship	6000
cat	6000	truck	6000
deer	6000	-	-

### 4.2 数据可视化与分析

通过可视化训练样本可以发现，不同类别的图像在颜色分布、纹理特征和形状结构上存在显著差异。例如：
    飞机和船类图像通常具有清晰的结构性边缘
    猫和狗类图像包含丰富的纹理信息
    汽车和卡车类图像具有相似的轮式交通工具特征，容易混淆

类别分布统计显示，训练集中各类别样本数量均衡（各约4500-5000张），不存在类别不平衡问题，为模型训练提供了公平的基础。

### 4.3 预处理流程

步骤	  操作	                                        说明
清洗	  无缺失值、无损坏样本	                   CIFAR-10数据质量高，无需额外清洗
标注	  已包含标签	                          每张图像对应0-9的整数标签
归一化	  transforms.Normalize	                  按通道均值-标准差归一化至标准正态分布
数据增强   RandomCrop + RandomHorizontalFlip	  随机裁剪（padding=4）和随机水平翻转，仅训练集使用
划分	  训练集45000 / 验证集5000 / 测试集10000	从训练集中分出5000张作为验证集

## 五、模型设计与选择

### 5.1 基准模型（Baseline）：SimpleCNN

构建一个三层卷积神经网络作为基线模型，用于对比验证最终模型的性能提升。

网络结构：

层	类型	                       参数	                         输出尺寸
1	Conv2d + BN + ReLU	       3→32, kernel=3, padding=1	    32×32×32
2	MaxPool2d	                  2×2	                        16×16×32
3	Conv2d + BN + ReLU	         32→64, kernel=3, padding=1	    16×16×64
4	MaxPool2d	                  2×2	                        8×8×64
5	Conv2d + BN + ReLU	         64→128, kernel=3, padding=1	8×8×128
6	MaxPool2d	                    2×2	                        4×4×128
7	Flatten	                       -	                        2048
8	Linear + ReLU + Dropout         	2048→256	            256
9	Linear	                        256→10	                     10

激活函数：ReLU
归一化方法：Batch Normalization、
正则化：Dropout (p=0.5)

### 5.2 最终模型架构

选用ResNet18作为最终模型，ResNet由何恺明等人于2016年提出，通过引入残差连接有效解决了深层网络的梯度消失和退化问题。

网络结构特点：
1.残差块（Residual Block） ：每个残差块包含两个3×3卷积层，通过跳跃连接将输入直接加到输出上，实现恒等映射的学习
2.批归一化：每个卷积层后均跟BN层，加速收敛并提高泛化能力
3.全局平均池化：替代全连接层，减少参数量

针对CIFAR-10的适配修改：
将第一层卷积从 kernel=7, stride=2 改为 kernel=3, stride=1, padding=1
去掉初始最大池化层（CIFAR-10图像尺寸较小）
将全连接层输出由1000改为10

选择该架构的理论依据：
1.残差连接解决了深度网络的梯度消失问题，使得网络可以更深、表达能力更强
2.在ImageNet上预训练的ResNet在各类视觉任务中表现优异，证明其泛化能力强
3.参数量适中（约11M），在有限计算资源下仍可高效训练

## 六、实验与结果分析

### 6.1 实验环境
CPU	Intel Core i7-12700H
GPU	NVIDIA GeForce RTX 3060 (6GB)
内存	16GB DDR5
Python	3.11.9
深度学习框架	PyTorch 2.0.0
主要库	torchvision, numpy, matplotlib, seaborn, scikit-learn

### 6.2 评价指标

指标	计算公式	说明
准确率（Accuracy）	(TP+TN) / (TP+TN+FP+FN)	总体预测正确的比例
精确率（Precision）	TP / (TP+FP)	预测为正类中实际为正类的比例
召回率（Recall）	TP / (TP+FN)	实际为正类中被正确预测的比例
F1分数	2×(P×R)/(P+R)	精确率和召回率的调和平均
宏平均F1	(1/N)×ΣF1_i	所有类别F1的算术平均
其中TP、TN、FP、FN分别表示真正例、真负例、假正例、假负例。

### 6.3 超参数设置与调优

固定超参数：
超参数	取值	说明
Batch Size	128	兼顾训练速度和GPU内存
损失函数	CrossEntropyLoss	适用于多分类任务
优化器	Adam	自适应学习率，收敛稳定
学习率	0.001	Adam默认建议值
学习率调度	StepLR (step=10, gamma=0.1)	每10轮学习率衰减为原来的0.1
Epochs	20	充分收敛且避免过拟合

调参记录：
实验	调整内容	验证集准确率	结论
实验1	lr=0.01	85.3%	学习率过高，训练不稳定
实验2	lr=0.001	90.8%	收敛稳定，效果最佳
实验3	lr=0.0001	89.5%	收敛速度过慢
实验4	无数据增强	87.2%	数据增强有效提升泛化能力
实验5	有数据增强	91.2%	采用该配置作为最终模型

### 6.4 主要实验结果

6.4.1 训练过程对比
Epoch	SimpleCNN (Val Acc)	ResNet18 (Val Acc)
1	42.3%	55.8%
5	65.7%	78.2%
10	72.4%	86.5%
15	76.1%	89.8%
20	78.6%	91.2%

6.4.2 测试集最终结果
模型	测试准确率	宏平均F1	参数量
SimpleCNN (Baseline)	77.8%	0.775	1.2M
ResNet18 (Final)	90.6%	0.904	11.2M

6.4.3 损失曲线与精度曲线
损失曲线显示：
ResNet18的收敛速度明显快于SimpleCNN
SimpleCNN的验证损失在第15个epoch后出现轻微上升，存在过拟合风险
ResNet18的验证损失在整个训练过程中持续下降，泛化能力更好

精度曲线显示：
ResNet18在第6个epoch后已超过SimpleCNN的最佳性能
学习率调整后两个模型都出现了性能跳跃，说明合理的学习率调度策略至关重要
（此部分由代码绘制，展示训练/验证损失下降曲线和精度上升曲线，两模型对比）

### 6.5 可视化分析

6.5.1 混淆矩阵分析

混淆矩阵展示了ResNet18在测试集上各类别的分类情况：
识别较好的类别：automobile（汽车）、ship（船）、truck（卡车）——准确率均在93%以上
识别较差的类别：cat（猫）和dog（狗）——二者存在较多相互误判，主要原因是猫和狗的外观特征较为相似
鸟和飞机也存在少量混淆：背景中天空区域较多时可能产生误判

6.5.2 错误样本分析

通过分析预测错误的样本，发现主要错误类型包括：
相似外观混淆：如猫误判为狗、鹿误判为马
图像质量影响：部分图像过于模糊或光照不佳
姿态变化：非标准角度或裁切导致的识别困难

6.5.3 特征图可视化

对ResNet18第一层卷积核的输出特征图进行可视化，观察到：
不同卷积核学习到了不同的特征模式，如边缘检测、纹理提取、颜色响应等
浅层特征图保留了较多的空间结构信息
随着网络加深，特征图逐步抽象为更高层次的语义特征

（此部分由代码生成具体的可视化图像）

！！！代码和图片都在另一个notebook中！！！